In [1]:
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
print(X[:5])
print(y[:5])

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
[0 0 0 0 0]


In [2]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=10, test_size=0.2, stratify=y, shuffle=True)
print(len(X_train))

120


In [3]:
from torch import nn
import torch.nn.functional as F
#클래스 생성(기존 클래스 상속)
class Model(nn.Module):
    def __init__(self, input_dim):  #__init__초기화함수
        super(Model, self).__init__()
        self.layer1 = nn.Linear(input_dim, 50)  #(input nodes, output nodes)
        self.layer2 = nn.Linear(50,20)
        self.layer3 = nn.Linear(20,3)

    def forward(self, x):  #순전파
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = F.softmax(self.layer3(x), dim=0)  #dim=0 첫번째 차원(row), row단위로 softmax 적용
        return x

In [4]:
import torch
model = Model(X_train.shape[1])  # 초기화함수의 input_dim으로 변수개수가 전달됨
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # 최적화함수 정의
loss_fn = nn.CrossEntropyLoss()  #손실함수 정의
epochs =100

In [5]:
#넘파이배열로부터 텐서를 만들고
X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).long()
for epoch in range(1, epochs+1):
    print('Epoch:', epoch)
    y_pred = model(X_train)
    loss = loss_fn(y_pred, y_train)
    print('loss:', loss.item())

    optimizer.zero_grad()  #가중치 초기화
    loss.backward()  #가중치 수정계산
    optimizer.step()  #수정분 반영

Epoch: 1
loss: 1.0987379550933838
Epoch: 2
loss: 1.0979018211364746
Epoch: 3
loss: 1.0970587730407715
Epoch: 4
loss: 1.096281886100769
Epoch: 5
loss: 1.0954458713531494
Epoch: 6
loss: 1.0944061279296875
Epoch: 7
loss: 1.093289852142334
Epoch: 8
loss: 1.0922987461090088
Epoch: 9
loss: 1.0914162397384644
Epoch: 10
loss: 1.0906733274459839
Epoch: 11
loss: 1.0900322198867798
Epoch: 12
loss: 1.089503526687622
Epoch: 13
loss: 1.089021921157837
Epoch: 14
loss: 1.0885636806488037
Epoch: 15
loss: 1.0881603956222534
Epoch: 16
loss: 1.087802529335022
Epoch: 17
loss: 1.0875006914138794
Epoch: 18
loss: 1.0872550010681152
Epoch: 19
loss: 1.0870474576950073
Epoch: 20
loss: 1.0868507623672485
Epoch: 21
loss: 1.0866572856903076
Epoch: 22
loss: 1.0864921808242798
Epoch: 23
loss: 1.0863293409347534
Epoch: 24
loss: 1.0861656665802002
Epoch: 25
loss: 1.0860083103179932
Epoch: 26
loss: 1.0858019590377808
Epoch: 27
loss: 1.0855205059051514
Epoch: 28
loss: 1.0851757526397705
Epoch: 29
loss: 1.0848755836486816

In [6]:
X_test = torch.from_numpy(X_test).float()
pred = model(X_test)
pred[:5]

tensor([[2.0154e-08, 5.0324e-03, 2.0593e-04],
        [2.0627e-02, 5.6921e-07, 1.2965e-12],
        [1.2226e-08, 1.1715e-03, 3.9892e-04],
        [2.5144e-09, 5.2105e-04, 1.4138e-03],
        [1.7519e-06, 3.4289e-01, 3.5229e-06]], grad_fn=<SliceBackward0>)

In [7]:
import numpy as np
np.argmax(pred.data.numpy(), axis=1)

array([1, 0, 1, 2, 1, 2, 0, 2, 2, 0, 0, 1, 2, 2, 1, 0, 0, 1, 2, 0, 2, 2,
       2, 0, 0, 1, 1, 0, 1, 1])

In [8]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, np.argmax(pred.data.numpy(), axis=1))

1.0

In [9]:
torch.save(model, 'c:/data/iris/iris-torch.h5')

In [10]:
model2 = torch.load('c:/data/iris/iris-torch.h5', weights_only=False)
np.argmax(model2(X_test[0]).data.numpy())

np.int64(2)

In [13]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                   Param #
Model                                    --
├─Linear: 1-1                            250
├─Linear: 1-2                            1,020
├─Linear: 1-3                            63
Total params: 1,333
Trainable params: 1,333
Non-trainable params: 0

In [12]:
summary(model, input_size=(32, 4))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [32, 3]                   --
├─Linear: 1-1                            [32, 50]                  250
├─Linear: 1-2                            [32, 20]                  1,020
├─Linear: 1-3                            [32, 3]                   63
Total params: 1,333
Trainable params: 1,333
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.04
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.01
Estimated Total Size (MB): 0.02